# FeroConCap on our data — matched comparison (authors' real code)

Retrains **FeroConCap** (Zhao et al., *J. Cheminformatics* 2026) on **our ferroptosis
dataset**, using the **authors' own `model.py`** (cloned from
github.com/zyy555/FeroConCap) unchanged — FCGR + capsule net + reconstruction decoder
+ supervised contrastive, their exact recipe (Adam lr 0.01, 30 epochs, margin +
recon×0.0005 + SupCon×0.005). We generate FCGR for our sequences (raw-count 64×64,
their `figure,label` format) and evaluate on a **random split** vs our **gene-disjoint
external set** — the same protocol as the other baselines.

**Before running:** Runtime → **GPU**; ferro data on Drive (`Data/Ferro/…`, incl. the
`RCD/` folder). **FeroConCap now trains on the FULL training set (no per-gene cap)** so it is directly comparable to ESM3FerroCLF & FRP-XGBoost. Then **Run all** — the capsule net on ~130k FCGR images takes **~1.5–3 h** on GPU (use a high-RAM/faster GPU if offered). The last cell prints the two AUROCs.


In [ ]:
# deps + clone the authors' repo
!pip install -q biopython openpyxl scikit-learn
!rm -rf /content/FeroConCap && git clone -q --depth 1 https://github.com/zyy555/FeroConCap /content/FeroConCap
import sys, os; sys.path.insert(0, '/content/FeroConCap')
print('cloned:', os.listdir('/content/FeroConCap'))

In [ ]:
# Drive, device, paths
from google.colab import drive; drive.mount('/content/drive')
import re, gzip, io, glob, time, math
from pathlib import Path
import numpy as np, pandas as pd, torch
from torch.optim import Adam
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score, accuracy_score
device = 'cuda' if torch.cuda.is_available() else 'cpu'
assert device == 'cuda', 'Set Runtime -> Change runtime type -> GPU'
print('device:', torch.cuda.get_device_name(0))
ROOT = Path('/content/drive/MyDrive/JR_Ferro'); D = ROOT/'Data/Ferro'
OUTD = ROOT/'feroconcap'; OUTD.mkdir(parents=True, exist_ok=True)
assert D.exists(), f'ferro data not found at {D}'

In [ ]:
# reconstruct our ferro dataset — mirrors ETAP_Ferro_final_eval EXACTLY
# (death genes from external/negative/ [Drive] or RCD/ [fallback]; pathway-stratified
# 70/30 gene-disjoint split, seed 42 — the SAME split as ESM3FerroCLF & the baselines)
random_seed=42
VALID=set('ACDEFGHIKLMNPQRSTVWY'); MAXLEN=1500
EXCLUDE_NEG={'NLRP3','FTL','YY1AP1'}; EXCLUDE_POS={'NLRP3'}
PER_GENE_CAP=1500; HOLDOUT_FRAC=0.30
PATHWAY={'ANXA5':'Apoptosis','APAF1':'Apoptosis','BBC3':'Apoptosis','BCL2':'Apoptosis',
    'CASP3':'Apoptosis','CASP7':'Apoptosis','DFFB':'Apoptosis','FADD':'Apoptosis',
    'CASP8':'Necroptosis','MLKL':'Necroptosis','RIPK1':'Necroptosis','RIPK3':'Necroptosis',
    'TNFRSF1A':'Necroptosis','TRADD':'Necroptosis','ZBP1':'Necroptosis',
    'AIM2':'Pyroptosis','CASP1':'Pyroptosis','CASP4':'Pyroptosis','CASP5':'Pyroptosis',
    'GSDMD':'Pyroptosis','GSDME':'Pyroptosis','IL1B':'Pyroptosis'}
def cln(s): return ''.join(c for c in str(s).upper() if c in VALID)[:MAXLEN]
def gene_from_fname(name):
    s=re.sub(r'(\.xlsx)+$','',name); s=re.sub(r'^uniparc_','',s); return re.split(r'_AND_|_20\d\d',s)[0].strip('_')
def read_xlsx_any(p):
    raw=Path(p).read_bytes()
    if raw[:2]==b'\x1f\x8b': raw=gzip.decompress(raw)
    return pd.read_excel(io.BytesIO(raw),engine='openpyxl')
def parse_xlsx(path,label,pathway):
    df=read_xlsx_any(path); df.columns=[c.strip() for c in df.columns]
    col=next((c for c in df.columns if 'seq' in c.lower()), df.columns[-1])
    g=gene_from_fname(os.path.basename(str(path))); rows=[]
    for v in df[col]:
        s=cln(v)
        if len(s)>=10: rows.append({'gene':g,'pathway':pathway,'seq':s,'label':label})
    return rows
def read_fa(path,label):
    seqs,genes=[],[]; hdr,buf=None,[]
    def fl():
        if hdr is None: return
        m=re.search(r'gene=([^|]*)',hdr); g=m.group(1).split('_AND_')[0] if m else 'NA'
        s=cln(''.join(buf))
        if len(s)>=10: seqs.append(s); genes.append(g)
    for line in open(path):
        if line.startswith('>'): fl(); hdr,buf=line,[]
        else: buf.append(line.strip())
    fl(); return pd.DataFrame({'seq':seqs,'gene':genes,'label':label})

# --- death (hard-neg) genes: external/negative/ (Drive) or RCD/ (fallback) ---
neg_paths=sorted((D/'external'/'negative').glob('*.xlsx'))
src='external/negative'
if not neg_paths:
    neg_paths=[Path(p) for p in glob.glob(str(D/'RCD/*/*.xlsx*'))]; src='RCD'
assert neg_paths, f'No death-gene xlsx under {D}/external/negative or {D}/RCD — upload one of them'
print(f'death-gene files: {len(neg_paths)} from {src}')
recs=[]
for p in neg_paths:
    g=gene_from_fname(os.path.basename(str(p)))
    if g in EXCLUDE_NEG: continue
    recs+=parse_xlsx(p,0,PATHWAY.get(g,'Other'))
rcd=pd.DataFrame(recs); print('death genes parsed:', sorted(rcd.gene.unique()))
rng=np.random.default_rng(random_seed)
def cap(group): return group if len(group)<=PER_GENE_CAP else group.iloc[rng.permutation(len(group))[:PER_GENE_CAP]]
rcd=rcd.groupby('gene',group_keys=False).apply(cap).reset_index(drop=True)
# pathway-stratified 70/30 gene-disjoint split (seed 42)
gp=rcd[['gene','pathway']].drop_duplicates().sort_values('gene').reset_index(drop=True)
hold,trn=[],[]
for pw,sub in gp.groupby('pathway'):
    genes=sub['gene'].to_numpy(); order=np.random.default_rng(random_seed).permutation(len(genes))
    nh=max(1,round(len(genes)*HOLDOUT_FRAC)); hold+=list(genes[order[:nh]]); trn+=list(genes[order[nh:]])
trn,hold=set(trn),set(hold)
rcd_train=rcd[rcd.gene.isin(trn)][['seq','gene','label']]; rcd_hold=rcd[rcd.gene.isin(hold)][['seq','gene','label']]
print('hard-neg TRAIN genes:', sorted(trn)); print('hard-neg HOLDOUT genes:', sorted(hold))
# external positives (external/*.xlsx, drop NLRP3)
extp=[]
for p in sorted(D.glob('external/*.xlsx')):
    if gene_from_fname(os.path.basename(str(p))) in EXCLUDE_POS: continue
    extp+=parse_xlsx(p,1,'ferroptosis')
ext_pos=pd.DataFrame(extp)[['seq','gene','label']].groupby('gene',group_keys=False).apply(cap).reset_index(drop=True)
# assemble
train_df=pd.concat([read_fa(D/'ferro_pos_rep_seq.fasta',1),
                    read_fa(D/'neg_versions/neg_v2_train_127.fasta',0), rcd_train],ignore_index=True)
# FULL training set — NO per-gene cap. Matches ESM3FerroCLF & FRP-XGBoost training
# (~130k seqs) for an apples-to-apples comparison; capsule-net training is just slower.
ext_df=pd.concat([ext_pos,rcd_hold],ignore_index=True)
assert train_df.label.nunique()==2 and ext_df.label.nunique()==2, 'a split has only one class!'
print('TRAIN', len(train_df), '(%.1f%% pos, %d genes)'%(100*train_df.label.mean(),train_df.gene.nunique()))
print('EXTERNAL', len(ext_df), '(%.1f%% pos, %d genes)'%(100*ext_df.label.mean(),ext_df.gene.nunique()))

In [ ]:
# FCGR (raw-count 64x64, n-flake) + import THEIR CapsNet
import model as FCM                                   # authors' model.py (CapsNet)
print('imported authors CapsNet | USE_CUDA =', FCM.USE_CUDA)
AA='ACDEFGHIKLMNPQRSTVWY'; AIDX={a:i for i,a in enumerate(AA)}
VERT=np.array([[math.sin(2*math.pi*i/20),math.cos(2*math.pi*i/20)] for i in range(20)])
_r=math.sin(math.pi/20)/(math.sin(math.pi/20)+math.sin(math.pi/20+2*math.pi*5/20)); SF=1-_r
def fcgr(seq,grid=64):
    p=np.zeros(2); img=np.zeros((grid,grid),np.float32)
    for a in seq:
        j=AIDX.get(a)
        if j is None: continue
        p=p+SF*(VERT[j]-p)
        x=min(max(int((p[0]+1)/2*grid),0),grid-1); y=min(max(int((p[1]+1)/2*grid),0),grid-1)
        img[y,x]+=1
    return img
def fcgr_batch(seqs): return np.stack([fcgr(s) for s in seqs])[:,None,:,:]
t0=time.time()
Xtr=torch.tensor(fcgr_batch(train_df.seq.tolist()),dtype=torch.float32); ytr=torch.tensor(train_df.label.values)
Xex=torch.tensor(fcgr_batch(ext_df.seq.tolist()),dtype=torch.float32);   yex=ext_df.label.values
print('FCGR', tuple(Xtr.shape), tuple(Xex.shape), 'in %.0fs'%(time.time()-t0))

In [ ]:
# train with THEIR recipe (Adam lr 0.01, 30 epochs, their loss) + stabilisation
# On the full dataset the authors' raw recipe diverges to NaN (~epoch 16).
# We add gradient clipping + a NaN guard + keep the best (lowest-loss) weights.
# Model, loss and lr are otherwise the authors' — unchanged.
import copy, math
g=torch.Generator().manual_seed(42); perm=torch.randperm(len(ytr),generator=g)
n_te=int(0.2*len(perm)); te_idx,tr_idx=perm[:n_te],perm[n_te:]
loader=DataLoader(TensorDataset(Xtr[tr_idx],ytr[tr_idx]),batch_size=64,shuffle=True)
net=FCM.CapsNet().to(device)
opt=Adam(net.parameters(),lr=0.01,betas=(0.9,0.999))
EPOCHS=30
best_loss=float('inf'); best_state=copy.deepcopy(net.state_dict())
for ep in range(EPOCHS):
    net.train(); tot=0.0; nb=0; skipped=0
    for data,target in loader:
        target=torch.sparse.torch.eye(2).index_select(0,target.long()).to(device)
        data=data.to(device)
        opt.zero_grad()
        output,rec,masked=net(data)
        loss=net.loss(data,output,target,rec)
        if not torch.isfinite(loss):                       # bad forward -> skip BEFORE backward
            skipped+=1; continue
        loss.backward()
        gn=torch.nn.utils.clip_grad_norm_(net.parameters(),5.0)   # tame the explosion
        if not torch.isfinite(gn):                         # NaN/inf grad -> skip, do not corrupt weights
            opt.zero_grad(); skipped+=1; continue
        opt.step(); tot+=loss.item(); nb+=1
    avg=tot/max(nb,1)
    if nb and math.isfinite(avg) and avg<best_loss:
        best_loss=avg; best_state=copy.deepcopy(net.state_dict())
    if ep%3==0 or ep==EPOCHS-1:
        print(f'epoch {ep+1}/{EPOCHS}  loss {avg:.4f}'
              + (f'  ({skipped} batches skipped)' if skipped else ''), flush=True)
    if not (nb and math.isfinite(avg)):                    # whole epoch unusable -> stop
        print('  entire epoch non-finite — restoring best weights and stopping'); break
net.load_state_dict(best_state)                            # eval uses the best finite weights
print(f'\nLoaded best weights (train loss {best_loss:.4f})')


In [ ]:
# evaluate: random split vs external
def predict(X):
    net.eval(); out=[]
    with torch.no_grad():
        for i in range(0,len(X),64):
            o,_,_=net(X[i:i+64].to(device))
            vc=torch.sqrt((o**2).sum(2)).squeeze(-1)          # (B,2)
            out.append(torch.softmax(vc,dim=1)[:,1].cpu())
    return torch.cat(out).numpy()
def metrics(y,p):
    pred=(p>=.5).astype(int); tp=((pred==1)&(y==1)).sum(); fn=((pred==0)&(y==1)).sum()
    tn=((pred==0)&(y==0)).sum(); fp=((pred==1)&(y==0)).sum()
    return dict(auroc=round(float(roc_auc_score(y,p)),4),acc=round(float(accuracy_score(y,pred)),4),
               sens=round(float(tp/max(tp+fn,1)),4),spec=round(float(tn/max(tn+fp,1)),4),n=int(len(y)))
m_rand=metrics(ytr[te_idx].numpy(), predict(Xtr[te_idx]))
m_ext =metrics(yex, predict(Xex))
res=pd.DataFrame([{'model':'FeroConCap','split':'random',**m_rand},
                  {'model':'FeroConCap','split':'external',**m_ext}])
res.to_csv(OUTD/'feroconcap_panelE.csv', index=False)
print(res.to_string(index=False))
print('\n=== FeroConCap (authors code) on our data ===')
print('  random  :', m_rand)
print('  external:', m_ext)
print('\nDOWNLOAD & send me:', OUTD/'feroconcap_panelE.csv')

**Send me** `feroconcap_panelE.csv`. I'll add FeroConCap as a 4th bar in
Figure 3 Panel E (random vs external). Note: on our data FCGR carries far less signal
than on their curated benchmark, so the *random* AUROC will be well below their 0.95 —
that's the honest result of their method on our harder, homology-controlled data.